In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report 
import pandas as pd
from sklearn.model_selection import GridSearchCV

In [16]:
df = pd.read_csv('Cleaned_F1_Data.csv')
df.columns

Index(['DriverNumber', 'Driver', 'TeamName', 'Position', 'Points', 'Year',
       'GrandPrix', 'Round', 'QualPosition', 'Driver_AvgFinish_Season',
       'Team_AvgPoints_Season', 'PrevRaceFinish', 'Form_Last3',
       'AvgFinish_AtGP_PastYears', 'AvgQual_AtGP_PastYears', 'norm_Form_Last3',
       'norm_AvgFinishAtGP', 'norm_AvgQualAtGP', 'Drivers_Confidence',
       'DriverID', 'TeamID', 'GP_ID'],
      dtype='object')

In [17]:
features = ['DriverID', 'TeamID', 'GP_ID', 'Year', 'Round','QualPosition', 'Driver_AvgFinish_Season',
             'Team_AvgPoints_Season','PrevRaceFinish', 'Form_Last3','AvgFinish_AtGP_PastYears', 
             'AvgQual_AtGP_PastYears','Drivers_Confidence']

X= df[features]
y = df['Position']

X_train , X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}


In [23]:
#RandomForest Classifier with GridSearchCV
rf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf, rf_param_grid, cv=2, scoring='accuracy', n_jobs=-1, verbose=2)
rf_grid.fit(X_train, y_train)

print("Best RandomForest params:", rf_grid.best_params_)
rf_best = rf_grid.best_estimator_

# Predict and evaluate
rf_pred = rf_best.predict(X_test)
# print("--- RandomForest Classification Report ---")
# print(classification_report(y_test, rf_pred))

Fitting 2 folds for each of 81 candidates, totalling 162 fits
Best RandomForest params: {'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


In [24]:
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0]
    }


y_train_zero = y_train - 1
y_test_zero = y_test - 1

In [25]:
#XGBoost Classifier with GridSearchCV
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_grid = GridSearchCV(xgb, xgb_param_grid, cv=2, scoring='accuracy', n_jobs=-1, verbose=1)
xgb_grid.fit(X_train, y_train_zero)

print("Best XGBoost params:", xgb_grid.best_params_)
xgb_best = xgb_grid.best_estimator_
xgb_pred = xgb_best.predict(X_test)
xgb_pred = xgb_pred + 1
# print("--- XGBoost Classification Report ---")
# print(classification_report(y_test, xgb_pred))

Fitting 2 folds for each of 243 candidates, totalling 486 fits


c:\Users\ashar\.vscode\First-ML-project\venv2\Lib\site-packages\xgboost\training.py:183: UserWarning: [16:17:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best XGBoost params: {'colsample_bytree': 0.85, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}


In [30]:
import pickle
# Save the models

pickle.dump(rf_best, open('rf_model.pkl', 'wb'))
pickle.dump(xgb_best, open('xgb_model.pkl', 'wb'))